# SLIIT IT3051 - EV Battery PHM Dual-Task System
## Week 2 - Task 2: Tree-Based Classifiers & Class Imbalance Mitigation (Critical Failure Risk)
### **Assigned Member: Person C**

---
### Overview & Objectives:
Address the 13.45:1 class imbalance using cost-sensitive learning (class_weight="balanced" and scale_pos_weight), Random Forest, and XGBoost.


### 0. Environment Setup & Data Partition Loading
Execute this cell to load the verified preprocessed matrices.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

def load_data():
    dataset_path = 'ev battery_failure  Dataset.csv'
    for p in [dataset_path, f'../{dataset_path}', f'../../{dataset_path}']:
        if os.path.exists(p):
            dataset_path = p
            break
            
    df = pd.read_csv(dataset_path)
    
    # 7 Domain Features
    df['temperature_spread'] = df['cell_temperature_max'] - df['cell_temperature_avg']
    df['efficiency_gap'] = (df['charge_efficiency'] - df['discharge_efficiency']).abs()
    df['health_loss_interaction'] = (df['battery_health_percent'] * df['capacity_loss_percent']) / 100.0
    df['resistance_per_1000_cycles'] = (df['internal_resistance'] / (df['cycle_count'] + 1.0)) * 1000.0
    df['c_rate_proxy'] = df['average_charge_power_kw'] / (df['battery_capacity_kwh'] + 1e-5)
    df['cell_voltage_spread'] = df['cell_voltage_std'] / (df['cell_voltage_avg'] + 1e-5)
    df['stress_index'] = df['aggressive_acceleration_score'] * df['hard_braking_score']
    
    id_cols = ['vehicle_id', 'battery_serial']
    target_rul = 'predicted_remaining_life_cycles'
    target_fail = 'battery_failure'
    
    excluded = id_cols + [target_rul, target_fail]
    features = [c for c in df.columns if c not in excluded]
    
    num_cols = df[features].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    
    # Task 1 Splits (RUL)
    df_t1 = df[df[target_rul].notnull()].copy()
    X1_tr, X1_te, y1_train, y1_test = train_test_split(df_t1[features], df_t1[target_rul], test_size=0.20, random_state=42)
    ct1 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X1_train = ct1.fit_transform(X1_tr)
    X1_test = ct1.transform(X1_te)
    
    # Task 2 Splits (Failure)
    X2_tr, X2_te, y2_train, y2_test = train_test_split(df[features], df[target_fail], test_size=0.20, random_state=42, stratify=df[target_fail])
    ct2 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X2_train = ct2.fit_transform(X2_tr)
    X2_test = ct2.transform(X2_te)
    
    feature_names = ct1.get_feature_names_out()
    return (X1_train, X1_test, y1_train, y1_test), (X2_train, X2_test, y2_train, y2_test), feature_names

print("Data Loader initialized successfully!")



In [ ]:
_, (X2_train, X2_test, y2_train, y2_test), feature_names = load_data()
print(f'Task 2 Train Shape: {X2_train.shape} | Test Shape: {X2_test.shape}')


### Task 1: Tree Classification Baseline: The Class Imbalance Problem
Train an unweighted Decision Tree and unweighted Random Forest Classifier. Evaluate Precision, Recall, and Confusion Matrix to show how accuracy masks missed failures.


In [ ]:
# TODO: Import DecisionTreeClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, average_precision_score

# TODO: Train unweighted DecisionTree and RandomForest

# TODO: Display confusion matrices and note the False Negatives



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- What is the Accuracy vs Recall for the unweighted Random Forest?
- How many actual battery failures were missed (False Negatives) by the unweighted model?


### Task 2: Cost-Sensitive Tree Learning (class_weight='balanced')
Train RandomForestClassifier with class_weight='balanced'. Evaluate the trade-off in Precision vs Recall.


In [ ]:
# TODO: Train RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

# TODO: Display updated confusion matrix and classification report



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- By how much did Recall increase when using class_weight='balanced'?
- What happened to Precision, and why is this an acceptable operational trade-off in battery safety?


### Task 3: Extreme Gradient Boosting with scale_pos_weight
Train HistGradientBoostingClassifier or XGBClassifier with scale_pos_weight = 13.45 (ratio of normal to failure instances).


In [ ]:
# TODO: Train cost-sensitive gradient boosting model
from sklearn.ensemble import HistGradientBoostingClassifier

# TODO: Fit model on X2_train, y2_train with class weights

# TODO: Evaluate test metrics (Recall, PR-AUC, ROC-AUC)



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- How does cost-sensitive boosting perform compared to balanced Random Forest?
- What are the top failure-driving features identified by the boosting model?


### Task 4: Precision-Recall & ROC Curve Comparison
Plot side-by-side ROC Curves and Precision-Recall Curves comparing all evaluated classification models.


In [ ]:
# TODO: Calculate predict_proba for each model

# TODO: Plot ROC Curves and Precision-Recall Curves



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- Why is the PR-AUC curve visually more informative than the ROC curve for a 13.45:1 imbalanced dataset?
- Which model achieved the highest PR-AUC score?
